## **DOWNLOADING COURT DECISIONS**

This notebook is designed to download court decisions from a specified source using Selenium for web automation. It includes configurations for download management and progress tracking.

### **Importing necessary libraries**

In [1]:
from selenium import webdriver
from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time
import os
import json
from datetime import datetime

### **Definining file/operation configurations**

This section contains file and operation configurations such as download directory, progress tracking file path, retry settings and ensures download directory exists


In [2]:
# Yapılandırma
download_dir = "/Users/beyzaasan/Projects/HukukPusulasi/hukukPusulasi-veri/Kararlar" # your-path-here
progress_file = "indirme_ilerleme.json"
max_retries = 3
retry_delay = 5

os.makedirs(download_dir, exist_ok=True)

### **Progress track**

Progress track section handles saving and loading progress information to resume downloads from where they left off


In [3]:
def load_progress():
    """İlerleme dosyasını yükle"""
    if os.path.exists(progress_file):
        with open(progress_file, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {"downloaded_files": [], "last_page": 0, "last_row": 0, "total_downloaded": 0}

def save_progress(progress):
    """İlerlemeyi kaydet"""
    with open(progress_file, 'w', encoding='utf-8') as f:
        json.dump(progress, f, ensure_ascii=False, indent=2)

In [4]:
def get_pdf_filename():
    """Son indirilen PDF'in adını al"""
    files = [f for f in os.listdir(download_dir) if f.endswith('.pdf')]
    if not files:
        return None
    latest = max([os.path.join(download_dir, f) for f in files], key=os.path.getmtime)
    return os.path.basename(latest)

### **Driver initiation**

Driver initiation section handles the initialization of the Chrome WebDriver with specific download preferences and configurations


In [5]:
def init_driver():
    """WebDriver'ı başlat"""
    chrome_options = Options()
    prefs = {
        "download.default_directory": download_dir,
        "download.prompt_for_download": False,
        "plugins.always_open_pdf_externally": True
    }
    chrome_options.add_experimental_option("prefs", prefs)
    driver = webdriver.Chrome(options=chrome_options)
    return driver, WebDriverWait(driver, 10)

### **Search website**

Search website section handles the search functionality on the UYAP website by performing a detailed search for consumer-related cases across all legal categories


In [6]:
def perform_search(driver, wait):
    """Arama işlemini gerçekleştir"""
    driver.get("https://emsal.uyap.gov.tr")
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Sayfa yükleniyor...")
    time.sleep(3)
    
    try:
        detayli_arama = wait.until(EC.element_to_be_clickable((By.LINK_TEXT, "Detaylı Arama")))
        detayli_arama.click()
        time.sleep(2)
    except:
        print("Detaylı Arama zaten açık...")
    
    arama_input = wait.until(EC.presence_of_element_located((By.ID, "arananDetail")))
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", arama_input)
    time.sleep(0.5)
    driver.execute_script("arguments[0].click();", arama_input)
    driver.execute_script("arguments[0].value = 'tüketici';", arama_input)
    driver.execute_script("""
        var event = new Event('input', { bubbles: true });
        arguments[0].dispatchEvent(event);
    """, arama_input)
    time.sleep(1)
    
    hukuk_dropdown_button = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "button[data-id='hukuk']")
    ))
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", hukuk_dropdown_button)
    time.sleep(0.5)
    driver.execute_script("arguments[0].click();", hukuk_dropdown_button)
    time.sleep(1)
    
    hepsini_sec_button = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "button.bs-select-all")
    ))
    driver.execute_script("arguments[0].click();", hepsini_sec_button)
    time.sleep(1)
    driver.execute_script("arguments[0].click();", hukuk_dropdown_button)
    time.sleep(1)
    
    ara_button = wait.until(EC.element_to_be_clickable((By.ID, "detaylıAramaG")))
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", ara_button)
    time.sleep(1)
    driver.execute_script("arguments[0].click();", ara_button)
    time.sleep(3)
    
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr")))
    time.sleep(2)
    
    try:
        page_size_select = wait.until(EC.presence_of_element_located((
            By.XPATH, "//label/select"
        )))
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", page_size_select)
        Select(page_size_select).select_by_visible_text("100")
        time.sleep(3)
        print("✓ Sayfa başına 100 sonuç ayarlandı")
    except:
        print("⚠ Sayfa boyutu ayarlanamadı")

### **Navigate where you left in progress**

- **Navigate to a specific page**
  
- This section contains a function that navigates through pages of search results
- The function takes a target page number and clicks through the pagination until reaching the desired page, handling any errors that occur during navigation


In [7]:
def navigate_to_page(driver, wait, target_page):
    """Belirli bir sayfaya git"""
    if target_page <= 1:
        return True
    
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Sayfa {target_page}'ye gidiliyor...")
    for page in range(2, target_page + 1):
        try:
            tbody = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody")))
            old_first_row = tbody.find_elements(By.CSS_SELECTOR, "tr")[0]
            
            next_button = None
            for selector in [
                "#detayAramaSonuclar_next",
                "#detayAramaSonuclar_paginate a.next",
                "a.paginate_button.next",
            ]:
                buttons = driver.find_elements(By.CSS_SELECTOR, selector)
                if buttons:
                    next_button = buttons[0]
                    break
            
            if not next_button:
                return False
            
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", next_button)
            driver.execute_script("arguments[0].click();", next_button)
            
            WebDriverWait(driver, 15).until(EC.staleness_of(old_first_row))
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr")))
            time.sleep(1)
            
        except Exception as e:
            print(f"⚠ Sayfa {page}'ye giderken hata: {str(e)}")
            return False
    
    return True

### **Download pdf (included retry)**

- Attempts to download PDF multiple times in case of failure 
- Clicks on row to open details 
- Finds and clicks PDF download button 
- Checks for new file in download directory
- Returns filename if successful, None if failed
- Handles errors by going back and retrying

In [8]:
def download_pdf_with_retry(driver, wait, row, row_index):
    """PDF indirmeyi yeniden deneme mekanizmasıyla gerçekleştir"""
    for attempt in range(max_retries):
        try:
            row.click()
            time.sleep(2)
            
            pdf_button = wait.until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR, "a.btn.btn-light-primary[onclick*='kararSavePdf']")
            ))
            
            old_files = set(os.listdir(download_dir))
            pdf_button.click()
            time.sleep(3)
            
            # Yeni dosya oluştuğunu kontrol et
            for _ in range(10):
                new_files = set(os.listdir(download_dir)) - old_files
                if new_files:
                    driver.back()
                    time.sleep(2)
                    return list(new_files)[0]
                time.sleep(1)
            
            driver.back()
            time.sleep(2)
            return None
            
        except Exception as e:
            print(f"  ⚠ Deneme {attempt + 1}/{max_retries} başarısız: {str(e)}")
            try:
                driver.back()
                time.sleep(2)
                # Satırları yeniden bul
                rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
                if row_index < len(rows):
                    row = rows[row_index]
            except:
                pass
            
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                return None
    
    return None

### **Main Function**

- Loads the progress from previous runs
- Initializes the web driver
- Performs the initial search
- Navigates to the last processed page
- Iterates through pages and rows to download PDFs
- Handles errors and retries
- Updates progress after each download

In [9]:
def main():
    progress = load_progress()
    print(f"\n{'='*60}")
    print(f"İndirme devam ediyor...")
    print(f"Son durum: Sayfa {progress['last_page']}, Satır {progress['last_row']}")
    print(f"Toplam indirilen: {progress['total_downloaded']} PDF")
    print(f"{'='*60}\n")
    
    driver = None
    try:
        driver, wait = init_driver()
        perform_search(driver, wait)
        
        # Kaldığı sayfaya git
        if progress['last_page'] > 1:
            if not navigate_to_page(driver, wait, progress['last_page']):
                print("⚠ Sayfaya gidilemedi, baştan başlanıyor...")
                progress['last_page'] = 1
                progress['last_row'] = 0
        
        current_page = progress['last_page'] if progress['last_page'] > 0 else 1
        
        while True:
            print(f"\n[{datetime.now().strftime('%H:%M:%S')}] === Sayfa {current_page} işleniyor ===")
            time.sleep(3)
            
            rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
            if not rows:
                print("⚠ Satır bulunamadı!")
                break
            
            print(f"Toplam {len(rows)} kayıt bulundu")
            
            # Kaldığı satırdan devam et
            start_row = progress['last_row'] if current_page == progress['last_page'] else 0
            
            for i in range(start_row, len(rows)):
                row = rows[i]
                try:
                    cells = row.find_elements(By.CSS_SELECTOR, "td")
                    if len(cells) >= 5:
                        status_cell = cells[4]
                        WebDriverWait(driver, 5).until(
                            lambda d: (status_cell.get_attribute("innerText") or "").strip() != ""
                        )
                        status = (status_cell.get_attribute("innerText") or "").strip()
                        
                        print(f"[{i+1}/{len(rows)}] Durum: {status}")
                        
                        if status == "KESİNLEŞMEDİ":
                            print("  → Kesinleşmedi, atlanıyor...")
                            continue
                        
                        filename = download_pdf_with_retry(driver, wait, row, i)
                        
                        if filename:
                            progress['total_downloaded'] += 1
                            progress['downloaded_files'].append(filename)
                            progress['last_page'] = current_page
                            progress['last_row'] = i + 1
                            save_progress(progress)
                            print(f"  ✓ İndirildi: {filename} (Toplam: {progress['total_downloaded']})")
                        else:
                            print(f"  ✗ İndirilemedi, devam ediliyor...")
                        
                        # Satırları yeniden bul
                        rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
                        
                except Exception as e:
                    print(f"  ✗ Hata: {str(e)}, devam ediliyor...")
                    try:
                        driver.back()
                        time.sleep(2)
                        rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
                    except:
                        pass
                    continue
            
            # Sonraki sayfaya geç
            try:
                tbody = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody")))
                old_first_row = tbody.find_elements(By.CSS_SELECTOR, "tr")[0]
                
                next_button = None
                for selector in [
                    "#detayAramaSonuclar_next",
                    "#detayAramaSonuclar_paginate a.next",
                    "a.paginate_button.next",
                ]:
                    buttons = driver.find_elements(By.CSS_SELECTOR, selector)
                    if buttons:
                        next_button = buttons[0]
                        break
                
                if not next_button:
                    print("\n✓ Tüm sayfalar tamamlandı!")
                    break
                
                classes = next_button.get_attribute("class") or ""
                if "disabled" in classes:
                    print("\n✓ Tüm sayfalar tamamlandı!")
                    break
                
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", next_button)
                driver.execute_script("arguments[0].click();", next_button)
                current_page += 1
                progress['last_page'] = current_page
                progress['last_row'] = 0
                save_progress(progress)
                
                WebDriverWait(driver, 15).until(EC.staleness_of(old_first_row))
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr")))
                time.sleep(1)
                
            except Exception as e:
                print(f"\n⚠ Sayfa geçişi başarısız: {str(e)}")
                break
        
        print(f"\n{'='*60}")
        print(f"✓ İşlem tamamlandı!")
        print(f"✓ Toplam {progress['total_downloaded']} PDF indirildi")
        print(f"✓ Dosyalar: {download_dir}")
        print(f"{'='*60}\n")
        
    except KeyboardInterrupt:
        print("\n\n⚠ Kullanıcı tarafından durduruldu")
        print(f"İlerleme kaydedildi. Toplam: {progress['total_downloaded']} PDF")
        print("Scripti tekrar çalıştırarak kaldığı yerden devam edebilirsiniz.")
        
    except Exception as e:
        print(f"\n⚠ Kritik hata: {str(e)}")
        print(f"İlerleme kaydedildi. Scripti tekrar çalıştırın.")
        
    finally:
        if driver:
            driver.quit()
        save_progress(progress)

In [12]:
if __name__ == "__main__":
    main()


İndirme devam ediyor...
Son durum: Sayfa 17, Satır 8
Toplam indirilen: 643 PDF

[13:09:58] Sayfa yükleniyor...
✓ Sayfa başına 100 sonuç ayarlandı
[13:10:17] Sayfa 17'ye gidiliyor...

[13:10:44] === Sayfa 17 işleniyor ===
Toplam 100 kayıt bulundu
[9/100] Durum: KESİNLEŞTİ
  ⚠ Deneme 1/3 başarısız: HTTPConnectionPool(host='localhost', port=57317): Read timed out. (read timeout=120)
  ⚠ Deneme 2/3 başarısız: Message: stale element reference: stale element not found
  (Session info: chrome=141.0.7390.76); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#staleelementreferenceexception
Stacktrace:
0   chromedriver                        0x0000000102fcf5f0 cxxbridge1$str$ptr + 2894960
1   chromedriver                        0x0000000102fc752c cxxbridge1$str$ptr + 2861996
2   chromedriver                        0x0000000102aed5ec _RNvCs47EqcsrPRmA_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 74324
3   chromedriver     